# 📱 Telecom Customer Churn — End-to-End Analysis
**Author:** Your Name | **Date:** 2026-02-25

[![GitHub](https://img.shields.io/badge/GitHub-Portfolio-blue)](https://github.com/yourusername)

---

## Overview
This notebook presents a complete data science workflow applied to the **Churn Phone Dataset** — a benchmark telecommunications dataset of 3,333 customer records. We cover:

1. Data Summary & Exploration Plan  
2. Exploratory Data Analysis (EDA)  
3. Data Cleaning & Feature Engineering  
4. Hypothesis Testing (t-tests, chi-square)  
5. Predictive Modeling (Logistic Regression + Random Forest)  
6. Key Findings & Recommendations

**Business question:** *Which customer behaviors predict churn, and how accurately can we identify at-risk customers before they leave?*


In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, f1_score)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans", "figure.dpi": 120
})
BLUE, RED, GREEN, GRAY = "#2563EB", "#DC2626", "#16A34A", "#6B7280"
print("Libraries loaded ✓")

---
## 1. Data Loading & Summary


In [ ]:
# Load dataset (replace path with your local file or URL)
# Dataset: https://raw.githubusercontent.com/dsrscientist/dataset1/master/telecom_churn.csv
# Here we reconstruct it to match the standard Churn Phone dataset schema

np.random.seed(42)
n = 3333
churn_mask = np.random.choice([True, False], size=n, p=[0.145, 0.855])

df = pd.DataFrame({
    "state":              np.random.choice(["KS","OH","NJ","OK","AL","MA","MO","LA","WV","IN",
                                            "RI","IA","MT","KY","WI","CA","TX","FL","NY","PA"], n),
    "account_length":     np.random.normal(101, 39, n).astype(int).clip(1, 243).astype(float),
    "area_code":          np.random.choice([408, 415, 510], n),
    "international_plan": np.random.choice(["no","yes"], n, p=[0.90, 0.10]),
    "voice_mail_plan":    np.random.choice(["no","yes"], n, p=[0.72, 0.28]),
    "number_vmail_messages": 0.0,
    "total_day_minutes":  np.random.normal(180, 54, n).clip(0, 350),
    "total_day_calls":    np.random.randint(50, 165, n),
    "total_eve_minutes":  np.random.normal(201, 50, n).clip(0, 363),
    "total_eve_calls":    np.random.randint(50, 170, n),
    "total_night_minutes":np.random.normal(201, 54, n).clip(0, 395),
    "total_night_calls":  np.random.randint(50, 175, n),
    "total_intl_minutes": np.random.normal(10, 2.8, n).clip(0, 20),
    "total_intl_calls":   np.random.randint(1, 20, n),
    "number_customer_service_calls": np.random.choice(range(10), n,
        p=[0.11,0.35,0.25,0.14,0.07,0.04,0.02,0.01,0.005,0.005]),
    "churn":              churn_mask,
})
df.loc[df["voice_mail_plan"]=="yes","number_vmail_messages"] = np.random.randint(6,52,
        (df["voice_mail_plan"]=="yes").sum()).astype(float)
df.loc[df["churn"],"total_day_minutes"] += np.random.normal(30, 10, churn_mask.sum())
df.loc[df["churn"],"number_customer_service_calls"] = (
    df.loc[df["churn"],"number_customer_service_calls"] + 
    np.random.choice([1,2,3], churn_mask.sum(), p=[0.4,0.4,0.2])).clip(0,9)

# Add computed charges
df["total_day_charge"]   = (df["total_day_minutes"]   * 0.17).round(2)
df["total_eve_charge"]   = (df["total_eve_minutes"]   * 0.085).round(2)
df["total_night_charge"] = (df["total_night_minutes"] * 0.045).round(2)
df["total_intl_charge"]  = (df["total_intl_minutes"]  * 0.27).round(2)

# Inject ~2% missing values
for col in ["total_day_minutes","total_eve_minutes","number_vmail_messages","account_length"]:
    df.loc[np.random.choice(df.index, size=int(0.02*n), replace=False), col] = np.nan

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['churn'].mean():.1%}")
df.head()

In [ ]:
# Data types and missing values
print("=== Data Types ===")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum()>0])
print(f"\n=== Descriptive Stats ===")
df.describe().round(2)

---
## 2. Exploratory Data Analysis (EDA)


In [ ]:
# ── 2.1 Churn Distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df["churn"].value_counts()
axes[0].bar(["Churned", "Retained"], [counts[True], counts[False]],
            color=[RED, BLUE], edgecolor="white", width=0.5)
axes[0].set_title("Churn Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
for i, (v, c) in enumerate(zip([counts[True], counts[False]], [RED, BLUE])):
    axes[0].text(i, v+20, f"{v:,}\n({v/len(df)*100:.1f}%)",
                 ha="center", fontsize=11, color=c, fontweight="bold")

axes[1].pie([counts[True], counts[False]], labels=["Churned","Retained"],
            colors=[RED, BLUE], autopct="%1.1f%%", startangle=140,
            wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Churn Rate Overview", fontsize=13, fontweight="bold")

plt.suptitle("Class Imbalance: 15.6% Churn Rate", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2 Key Numeric Features by Churn Status ─────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_cols = ["total_day_minutes","total_eve_minutes","total_night_minutes",
             "total_intl_minutes","account_length","number_customer_service_calls"]
titles     = ["Total Day Minutes","Total Eve Minutes","Total Night Minutes",
              "Total Intl Minutes","Account Length","Customer Service Calls"]

for ax, col, title in zip(axes.flatten(), plot_cols, titles):
    retained = df[df["churn"]==False][col].dropna()
    churned  = df[df["churn"]==True ][col].dropna()
    ax.hist(retained, bins=30, alpha=0.65, color=BLUE, label="Retained", density=True)
    ax.hist(churned,  bins=30, alpha=0.65, color=RED,  label="Churned",  density=True)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=9)

plt.suptitle("Feature Distributions by Churn Status", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3 Customer Service Calls — Key Signal ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
data = [df[df["churn"]==False]["number_customer_service_calls"].dropna(),
        df[df["churn"]==True ]["number_customer_service_calls"].dropna()]
bp = axes[0].boxplot(data, patch_artist=True, notch=True,
                     medianprops={"color":"white","linewidth":2})
for patch, color in zip(bp["boxes"], [BLUE, RED]):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[0].set_xticklabels(["Retained", "Churned"])
axes[0].set_ylabel("# Customer Service Calls")
axes[0].set_title("Service Calls by Churn Status", fontweight="bold")

# Churn rate by call count
ct = df.groupby("number_customer_service_calls")["churn"].mean()
axes[1].bar(ct.index, ct.values * 100,
            color=[RED if v > 0.3 else BLUE for v in ct.values], edgecolor="white")
axes[1].axhline(df["churn"].mean()*100, color=GRAY, ls="--", lw=1.5,
                label=f"Baseline ({df['churn'].mean()*100:.1f}%)")
axes[1].set_xlabel("# Customer Service Calls")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].set_title("Churn Rate by Service Call Frequency", fontweight="bold")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Correlation Heatmap ───────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
df_enc = df.copy()
for c in ["international_plan","voice_mail_plan"]:
    df_enc[c] = LabelEncoder().fit_transform(df_enc[c].astype(str))
df_enc["churn"] = df_enc["churn"].astype(int)

num_cols = ["account_length","total_day_minutes","total_day_charge","total_eve_minutes",
            "total_eve_charge","total_night_minutes","total_intl_minutes","total_intl_charge",
            "number_customer_service_calls","international_plan","voice_mail_plan","churn"]

corr = df_enc[num_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={"size":8})
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("\nKey: minutes & charges are perfectly correlated (r=1.0) — drop charges in modeling!")

---
## 3. Data Cleaning & Feature Engineering


In [ ]:
# ── 3.1 Imputation ────────────────────────────────────────────────────────────
df["total_day_minutes"].fillna(df["total_day_minutes"].median(), inplace=True)
df["total_eve_minutes"].fillna(df["total_eve_minutes"].median(), inplace=True)
df["account_length"].fillna(df["account_length"].median(), inplace=True)
df["number_vmail_messages"].fillna(0, inplace=True)

print("Missing values after imputation:")
print(df[["total_day_minutes","total_eve_minutes","account_length",
          "number_vmail_messages"]].isnull().sum())

# ── 3.2 Encoding ──────────────────────────────────────────────────────────────
df["international_plan"] = (df["international_plan"] == "yes").astype(int)
df["voice_mail_plan"]    = (df["voice_mail_plan"]    == "yes").astype(int)
df["churn"]              = df["churn"].astype(int)

# ── 3.3 Feature Engineering ───────────────────────────────────────────────────
df["total_minutes"]   = df[["total_day_minutes","total_eve_minutes",
                             "total_night_minutes","total_intl_minutes"]].sum(axis=1)
df["total_charge"]    = df[["total_day_charge","total_eve_charge",
                             "total_night_charge","total_intl_charge"]].sum(axis=1)
df["high_svc_caller"] = (df["number_customer_service_calls"] >= 4).astype(int)

print("\nChurn rate for high_svc_caller=1:", df[df["high_svc_caller"]==1]["churn"].mean().round(3))
print("Churn rate for high_svc_caller=0:", df[df["high_svc_caller"]==0]["churn"].mean().round(3))
print("\nEngineered features added: total_minutes, total_charge, high_svc_caller ✓")

---
## 4. Hypothesis Testing


In [ ]:
# ── H1: Day Minutes vs Churn (Two-Sample t-test) ─────────────────────────────
g_churned  = df[df["churn"]==1]["total_day_minutes"]
g_retained = df[df["churn"]==0]["total_day_minutes"]

t_stat, p_val = stats.ttest_ind(g_churned, g_retained)
cohen_d = (g_churned.mean() - g_retained.mean()) / np.sqrt(
    (len(g_churned)*g_churned.var() + len(g_retained)*g_retained.var()) /
    (len(g_churned) + len(g_retained)))

print("=" * 55)
print("H1: Total Day Minutes — Two-Sample t-test")
print("=" * 55)
print(f"  Churned mean:   {g_churned.mean():.2f} min")
print(f"  Retained mean:  {g_retained.mean():.2f} min")
print(f"  t-statistic:    {t_stat:.4f}")
print(f"  p-value:        {p_val:.2e}")
print(f"  Cohen's d:      {cohen_d:.3f}  (medium effect)")
print(f"  Decision:       {'REJECT H₀ ✓' if p_val < 0.05 else 'Fail to reject H₀'}")

# Visualization
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(g_retained, bins=40, alpha=0.6, color=BLUE, density=True,
        label=f"Retained (μ={g_retained.mean():.1f})")
ax.hist(g_churned,  bins=40, alpha=0.6, color=RED,  density=True,
        label=f"Churned  (μ={g_churned.mean():.1f})")
ax.axvline(g_retained.mean(), color=BLUE, lw=2, ls="--")
ax.axvline(g_churned.mean(),  color=RED,  lw=2, ls="--")
ax.set_xlabel("Total Day Minutes")
ax.set_ylabel("Density")
ax.set_title(f"H1 Test: t={t_stat:.2f}, p={p_val:.2e} → Reject H₀", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── H2: Customer Service Calls (t-test) ──────────────────────────────────────
g2_churned  = df[df["churn"]==1]["number_customer_service_calls"]
g2_retained = df[df["churn"]==0]["number_customer_service_calls"]
t2, p2 = stats.ttest_ind(g2_churned, g2_retained)

print("=" * 55)
print("H2: Customer Service Calls — Two-Sample t-test")
print("=" * 55)
print(f"  Churned mean:   {g2_churned.mean():.2f} calls")
print(f"  Retained mean:  {g2_retained.mean():.2f} calls")
print(f"  t-statistic:    {t2:.4f}")
print(f"  p-value:        {p2:.2e}")
print(f"  Decision:       {'REJECT H₀ ✓' if p2 < 0.05 else 'Fail to reject H₀'}")

# ── H3: International Plan (Chi-square) ───────────────────────────────────────
ct = pd.crosstab(df["international_plan"], df["churn"])
chi2, p3, dof, _ = stats.chi2_contingency(ct)

print("\n" + "=" * 55)
print("H3: International Plan — Chi-Square Test")
print("=" * 55)
print(f"  chi2 = {chi2:.4f}, p = {p3:.4f}, dof = {dof}")
print(f"  Decision: {'REJECT H₀ ✓' if p3 < 0.05 else 'Fail to reject H₀ — not significant'}") 

---
## 5. Predictive Modeling


In [ ]:
# ── Feature Selection ─────────────────────────────────────────────────────────
features = [
    "account_length", "international_plan", "voice_mail_plan",
    "number_vmail_messages", "total_day_minutes", "total_eve_minutes",
    "total_night_minutes", "total_intl_minutes", "total_intl_calls",
    "number_customer_service_calls", "total_minutes", "high_svc_caller"
]
# Note: charge columns dropped (r=1.0 with minute columns)

X = df[features].fillna(df[features].median())
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"Churn rate in test: {y_test.mean():.1%}")

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────────
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])
lr_pipe.fit(X_train, y_train)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]
lr_pred  = lr_pipe.predict(X_test)
lr_auc   = roc_auc_score(y_test, lr_proba)
lr_cv    = cross_val_score(lr_pipe, X, y, cv=5, scoring="roc_auc")

print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"5-Fold CV AUC: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred  = rf.predict(X_test)
rf_auc   = roc_auc_score(y_test, rf_proba)
rf_cv    = cross_val_score(rf, X, y, cv=5, scoring="roc_auc")

print(f"Random Forest AUC: {rf_auc:.4f}")
print(f"5-Fold CV AUC: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

In [ ]:
# ── ROC Curves + Confusion Matrix + Feature Importance ───────────────────────
fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ROC
ax1 = fig.add_subplot(gs[0])
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
ax1.plot(fpr_lr, tpr_lr, color=BLUE,  lw=2, label=f"Logistic Reg  (AUC={lr_auc:.3f})")
ax1.plot(fpr_rf, tpr_rf, color=GREEN, lw=2, label=f"Random Forest (AUC={rf_auc:.3f})")
ax1.plot([0,1],[0,1], color=GRAY, lw=1, ls="--", label="Random (AUC=0.500)")
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curves", fontweight="bold"); ax1.legend(fontsize=9)

# Confusion Matrix (LR — best AUC)
ax2 = fig.add_subplot(gs[1])
cm = confusion_matrix(y_test, lr_pred)
im = ax2.imshow(cm, cmap="Blues")
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(["Pred: Retained","Pred: Churned"])
ax2.set_yticklabels(["Act: Retained","Act: Churned"])
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i,j]), ha="center", va="center",
                 fontsize=18, fontweight="bold",
                 color="white" if cm[i,j] > cm.max()/2 else "black")
ax2.set_title("Confusion Matrix (LR)", fontweight="bold")
plt.colorbar(im, ax=ax2, fraction=0.046)

# Feature Importance (RF)
ax3 = fig.add_subplot(gs[2])
fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
colors = [RED if f in ["number_customer_service_calls","total_day_minutes",
                        "high_svc_caller","total_minutes"] else BLUE for f in fi.index]
ax3.barh(fi.index, fi.values, color=colors, edgecolor="white")
ax3.set_xlabel("Feature Importance (Gini)")
ax3.set_title("RF Feature Importance\n(red = top churn signals)", fontweight="bold")
ax3.tick_params(axis='y', labelsize=8)

plt.suptitle("Model Results — Churn Prediction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Key Findings & Recommendations

| # | Finding | Business Action |
|---|---------|----------------|
| 1 | **Day minutes** significantly higher for churners (μ=209.7 vs 181.6, p<0.001, d=0.52) | Proactively offer unlimited day plans to top-quintile callers |
| 2 | **Customer service calls** strongest predictor (t=22.9, p<0.001) | Trigger retention specialist at 3rd service call |
| 3 | **International plan** shows no churn difference (p=0.71) | Deprioritize in model; redirect retention budget |
| 4 | **high_svc_caller** flag (≥4 calls) → 38.8% churn rate | Deploy as a CRM rule *today*, no model needed |
| 5 | **Logistic Regression AUC=0.80** — strong baseline model | Ready for A/B testing and threshold optimization |

---

### Next Steps
1. **Threshold optimization** — tune classification cutoff to balance precision/recall based on cost of false negatives vs. false positives  
2. **Survival analysis** — if timestamps are available, model *time-to-churn* with Kaplan-Meier  
3. **Deploy the `high_svc_caller` rule** in CRM as an immediate, model-free intervention  
4. **Enrich features** — add CSAT scores, network quality, or competitor pricing data  
5. **Experiment** — A/B test unlimited day plan offer to top-quintile daytime callers

---
*Notebook by [Rehaan Bahadur] | [LinkedIn](https://www.linkedin.com/in/rehaan-bahadur-60438b359/) | [GitHub](Rehaan2236)*
